In [6]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install rank_bm25
import sys, os, re, importlib, datetime, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"; DS = f"{BASE}/dataset_v2.3"

df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{DS}/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
if "small_version" not in df.columns:
    J = pd.read_parquet(f"{DS}/esci_electronics_v2.3_judgments.parquet")
    df["small_version"] = df["query_id"].map(J.groupby("query_id")["small_version"].max())
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"]) and not (ids["train"] & ids["val"])
val  = df[df.part == "val"].reset_index(drop=True)
test = df[df.part == "test"].reset_index(drop=True)
del df

CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
for d in (val, test):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

print("val q", val.query_id.nunique(), "| test q", test.query_id.nunique(), "(expect 1295)",
      "| test small_version q", test.loc[test.small_version == 1, "query_id"].nunique(), "(expect 507)")
print("GPU", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
val q 656 | test q 1295 (expect 1295) | test small_version q 507 (expect 507)
GPU Tesla T4


In [7]:
# ---- exactly the S4 BM25 (05_baselines): word statistics from the whole 51,318-product corpus
tokenize = lambda s: re.findall(r"[a-z0-9]+", str(s).lower())
corpus = pd.read_parquet(f"{DS}/esci_electronics_v2.3_corpus.parquet")
corpus_text = [C.product_to_text({"title": t, "brand": b, "color": c}, "full")
               for t, b, c in zip(corpus["product_title"], corpus["product_brand"], corpus["product_color"])]
bm25 = BM25Okapi([tokenize(t) for t in corpus_text])
pid2idx = {p: i for i, p in enumerate(corpus["product_id"])}

def bm25_scores(d, text_col=None):               # text_col kept only so Cell 3 runs unchanged
    out = np.zeros(len(d))
    for qid, idx in d.groupby("query_id").indices.items():
        ids = [pid2idx[p] for p in d["product_id"].values[idx]]
        out[idx] = bm25.get_batch_scores(tokenize(d["query"].values[idx[0]]), ids)
    return out
def ce_scores(path, d, qcol, tcol):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
    s = T.score_pairs(m, tok, d[qcol].tolist(), d[tcol].tolist(), 256, 128)
    del m; torch.cuda.empty_cache(); return s
def ndcg(d, scores):
    v = d[["query_id", "gain"]].copy(); v["score"] = scores; return evaluate(v)[0]["ndcg@10"]

ZS = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BM_TEXT, ZS_TEXT = "corpus_full", "text_full"
b = ndcg(val, bm25_scores(val)); z = ndcg(val, ce_scores(ZS, val, "query", ZS_TEXT))
s4 = pd.read_csv(f"{BASE}/results/baselines_val.csv")
b_s4 = s4.loc[s4.model == "BM25", "ndcg@10"].iloc[0]
print(f"BM25      val NDCG@10 = {b:.4f} | S4 saved value = {b_s4:.4f}")
print(f"zero-shot val NDCG@10 = {z:.4f} | S4 ≈ 0.912")
ok = abs(b - b_s4) <= 0.001 and abs(z - 0.912) <= 0.003
print("Baselines reproduce S4:", ok)
assert ok, "STOP: send me this output. Test NOT used."

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BM25      val NDCG@10 = 0.8722 | S4 saved value = 0.8722
zero-shot val NDCG@10 = 0.9122 | S4 ≈ 0.912
Baselines reproduce S4: True


In [8]:
FLAG = f"{BASE}/results/S10_TEST_USED.txt"; FORCE = False
assert FORCE or not os.path.exists(FLAG), "Test already used. Do not re-run (set FORCE=True only after a crash)."

def best(log_file, prefix):                      # best epoch on VALIDATION, normal formats only
    L = pd.read_csv(f"{BASE}/results/{log_file}"); L = L[L["style"].isin(C.STYLES)]
    L = L[L.run.str.fullmatch(rf"{prefix}__q_focused_cat(__s\d+)?")]
    ep = L.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
    return ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "epoch"]]
RUNS = {"augplus": best("s9e_augplus_val_log.csv", "minilm_l6_augplus"), "mix": best("s7_grid_val_log.csv", "minilm_l6")}
for g, b in RUNS.items(): print(g, b.values.tolist())

SUBSETS = {"full_test": np.ones(len(test), bool), "small_version": (test.small_version == 1).values}
rows, PQ = [], {}
def record(model, seed, fmt, scores):
    v = test[["query_id", "gain"]].copy(); v["score"] = scores
    for sub, mask in SUBSETS.items():
        mtr, pq = evaluate(v[mask])
        rows.append({"subset": sub, "model": model, "seed": seed, "format": fmt, **mtr})
        PQ[(sub, model, seed, fmt)] = pq.set_index("query_id")["ndcg@10"]

record("bm25", 0, "bm25", bm25_scores(test, BM_TEXT))
record("zeroshot", 0, "full_bullets", ce_scores(ZS, test, "query", ZS_TEXT))
for g, b in RUNS.items():
    for i, (_, r) in enumerate(b.iterrows()):
        path = f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}"
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for s in C.STYLES:
            record(g, i, s, T.score_pairs(m, tok, test["q_focused_cat"].tolist(), test[f"text_{s}"].tolist(), 256, 128))
        del m; torch.cuda.empty_cache(); print("scored", r.run)
open(FLAG, "w").write(f"test used {datetime.datetime.now()}\n")

R = pd.DataFrame(rows); R.to_csv(f"{BASE}/results/s10_final_test.csv", index=False)
for sub in SUBSETS:
    print(f"\n================ {sub} ================")
    main = R[(R.subset == sub) & R.format.isin(["bm25", "full_bullets"])]
    t = main.groupby("model")[["ndcg@10", "mrr", "p@5"]].agg(["mean", "std"]).reindex(["bm25", "zeroshot", "mix", "augplus"])
    print(t.round(4).to_string())
    f5 = R[(R.subset == sub) & R.model.isin(["mix", "augplus"]) & R.format.isin(C.STYLES)]
    print("5-format mean NDCG@10:", f5.groupby(["model", "seed"])["ndcg@10"].mean().groupby("model").agg(["mean", "std"]).round(4).to_dict("index"))
    avgq = lambda mod, fmt: pd.concat([v for (s, mm, sd, f), v in PQ.items() if s == sub and mm == mod and f == fmt], axis=1).mean(axis=1)
    a = avgq("augplus", "full_bullets")
    for name, other in [("BM25", avgq("bm25", "bm25")), ("zero-shot", avgq("zeroshot", "full_bullets")), ("mix", avgq("mix", "full_bullets"))]:
        d = a - other.loc[a.index]; nz = d[d != 0]
        print(f"augplus vs {name:9s}: {d.mean():+.4f} | better {int((d>0).sum())} / worse {int((d<0).sum())} | p = {wilcoxon(nz).pvalue:.2e}")

augplus [['minilm_l6_augplus__q_focused_cat', 3], ['minilm_l6_augplus__q_focused_cat__s43', 2], ['minilm_l6_augplus__q_focused_cat__s44', 3]]
mix [['minilm_l6__q_focused_cat', 3], ['minilm_l6__q_focused_cat__s43', 3], ['minilm_l6__q_focused_cat__s44', 2]]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_augplus__q_focused_cat


Loading weights:   0%|          | 0/105 [00:01<?, ?it/s]

...

scored minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:01<?, ?it/s]

scored minilm_l6__q_focused_cat__s44

================ full_test ================
         ndcg@10             mrr             p@5        
            mean     std    mean     std    mean     std
model                                                   
bm25      0.8440     NaN  0.8665     NaN  0.7351     NaN
zeroshot  0.8788     NaN  0.9039     NaN  0.7758     NaN
mix       0.9058  0.0008  0.9363  0.0026  0.7974  0.0010
augplus   0.9057  0.0009  0.9354  0.0010  0.7961  0.0017
5-format mean NDCG@10: {'augplus': {'mean': 0.9033, 'std': 0.001}, 'mix': {'mean': 0.9034, 'std': 0.0006}}
augplus vs BM25     : +0.0617 | better 530 / worse 249 | p = 2.99e-34
augplus vs zero-shot: +0.0269 | better 442 / worse 281 | p = 2.79e-14
augplus vs mix      : -0.0001 | better 288 / worse 339 | p = 2.94e-02

================ small_version ================
         ndcg@10             mrr             p@5        
            mean     std    mean     std    mean     std
model                                  

In [9]:
print(val.groupby("query_id").small_version.max().mean(), test.groupby("query_id").small_version.max().mean())

0.25914634146341464 0.3915057915057915
